In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Load data set

orders = pd.read_csv('extracted_data/olist_orders_dataset.csv')
customers = pd.read_csv('extracted_data/olist_customers_dataset.csv')
order_items = pd.read_csv('extracted_data/olist_order_items_dataset.csv')
products = pd.read_csv('extracted_data/olist_products_dataset.csv')
payments = pd.read_csv('extracted_data/olist_order_payments_dataset.csv')
sellers = pd.read_csv('extracted_data/olist_sellers_dataset.csv')
reviews = pd.read_csv('extracted_data/olist_order_reviews_dataset.csv')
category = pd.read_csv('extracted_data/product_category_name_translation.csv')


In [4]:
orders.shape

(99441, 8)

In [5]:
customers.shape

(99441, 5)

In [6]:
order_items.shape

(112650, 7)

In [7]:
products.shape

(32951, 9)

In [8]:
payments.shape

(103886, 5)

In [9]:
reviews.shape

(99224, 7)

In [10]:
sellers.shape

(3095, 4)

In [11]:
category.shape

(71, 2)

In [12]:
print(f"\n orders:{orders.shape}")


 orders:(99441, 8)


In [13]:
def quick_check(df,name):
    print(f"\n{'='*45}")
    print(f" {name}")
    print(f"shape: {df.shape}")
    print(f"\nMissing values:")
    missing = df.isnull().sum()[df.isnull().sum()>0]
    if len(missing) ==0:
        print('no missing values')
    else:
        print(missing)
    print(f"{'='*45}")

In [14]:
orders.isnull().sum()[orders.isnull().sum()>0]

order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

In [15]:
quick_check(orders,'orders')


 orders
shape: (99441, 8)

Missing values:
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64


In [16]:
quick_check(products,'products')



 products
shape: (32951, 9)

Missing values:
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


In [17]:
quick_check(reviews,'reviews')


 reviews
shape: (99224, 7)

Missing values:
review_comment_title      87656
review_comment_message    58247
dtype: int64


In [18]:
orders['order_purchase_timestamp']= pd.to_datetime(orders['order_purchase_timestamp']) 
orders['order_approved_at'] = pd.to_datetime(orders['order_approved_at'])
orders['order_delivered_carrier_date'] = pd.to_datetime(orders['order_delivered_carrier_date'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])

In [19]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB


In [20]:
products['product_category_name'] = products['product_category_name'].fillna('unknown')

In [21]:
# Dimensions fill by median.

for col in ['product_weight_g','product_length_cm','product_height_cm','product_width_cm']:
    products[col] = products[col].fillna(products[col].median())

In [22]:
products.drop(columns=['product_name_lenght','product_description_lenght','product_photos_qty'],inplace=True)

In [23]:
products.isnull().sum()

product_id               0
product_category_name    0
product_weight_g         0
product_length_cm        0
product_height_cm        0
product_width_cm         0
dtype: int64

In [24]:
reviews['review_comment_title'] = reviews['review_comment_title'].fillna('No Title')
reviews['review_comment_message'] = reviews['review_comment_message'].fillna('No comments')


In [25]:
reviews.isnull().sum()

review_id                  0
order_id                   0
review_score               0
review_comment_title       0
review_comment_message     0
review_creation_date       0
review_answer_timestamp    0
dtype: int64

In [29]:
# create a master dataset.

df = orders.merge(customers, how='left',on='customer_id')
df = df.merge(order_items, how='left', on = 'order_id')
df = df.merge(products,how='left', on ='product_id')
df = df.merge(payments, how='left',on= 'order_id')
df = df.merge(sellers, how='left',on='seller_id')
df = df.merge(category, how='left', on='product_category_name')
df=df.merge(reviews,how='left',on='order_id')

In [30]:
df.shape

(119143, 37)

In [32]:
# master dataset saved in to folder.
df.to_csv('extracted_data/master_dataset.csv',index=False)